In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Load data

In [ ]:
factories_df = pd.read_csv("./data/US Candy Distributors/Candy_Factories.csv")
products_df = pd.read_csv("./data/US Candy Distributors/Candy_Products.csv")
sales_df = pd.read_csv("./data/US Candy Distributors/Candy_Sales.csv", parse_dates=["Ship Date", "Order Date"])
targets_df = pd.read_csv("./data/US Candy Distributors/Candy_Targets.csv")
uszips_df = pd.read_csv("./data/US Candy Distributors/uszips.csv", dtype={"zip": str})

In [ ]:
candy_sales_dfs = {
    "Candy Factories": factories_df,
    "Candy Products": products_df,
    "Candy Sales": sales_df,
    "Candy Tragets": targets_df,
    "US zips": uszips_df
}

for name, df in candy_sales_dfs.items():
    print("=" * 20)
    print(name)
    print("=" * 20)

    print(df.dtypes)
    print("No. of rows:", df.shape[0])
    print("")
    print(df.head(2))
    print("")
    print("*" * 100)
    print("")


## Columns rename

In [ ]:
def rename_columns(df):
    return [col.lower().replace(" ", "_").replace("/", "_") for col in df.columns]

for df in candy_sales_dfs.values():
    df.columns = rename_columns(df)

## apply()

In [ ]:
sales_df.head()

### Calculate the Gross Profit %

```
Gross Profit % = (Sales - Cost) / Cost
```

In [ ]:
round((sales_df["sales"] - sales_df["cost"]) / sales_df["cost"] * 100, 2)

In [ ]:
sales_df.apply(lambda row: (row["sales"] - row["cost"]) / row["cost"] * 100, axis=1)

In [ ]:
def calc_gross_profit(row):
    sales = row["sales"]
    cost = row["cost"]

    gross_profit = (sales - cost) / cost * 100
    return round(gross_profit, 2)

sales_df.apply(calc_gross_profit, axis=1)

In [ ]:
sales_df["gross_profit_percent"] = sales_df.apply(calc_gross_profit, axis=1)

In [ ]:
sales_df.head()

In [ ]:
df_1 = sales_df[["sales", "units", "cost", "gross_profit"]]

df_1.apply(sum, axis=0)

In [ ]:
df_1.head()

In [ ]:
df_1["sales"] + df_1["units"] + df_1["cost"] + df_1["gross_profit"]

### Question: Find the month over month growth %

In [ ]:
sales_df["order_year"] = sales_df["order_date"].dt.year
sales_df["order_month"] = sales_df["order_date"].dt.month

sales_df.head()

In [ ]:
monthly_sales_df = sales_df.groupby(["order_year", "order_month"]).agg(total_sales=("sales", "sum"))
monthly_sales_df.head(15)

In [ ]:
prev_month_sales_df = monthly_sales_df.groupby(["order_year"]).shift(1).rename(columns={"total_sales": "prev_month_sales"})
prev_month_sales_df.head()

In [ ]:
monthly_sales_joined_df = monthly_sales_df.join(prev_month_sales_df)
monthly_sales_joined_df.head()

In [ ]:
def mom_growth(row):
    current_month_sales = row["total_sales"]
    previous_month_sales = row["prev_month_sales"]

    growth = (current_month_sales - previous_month_sales) / previous_month_sales * 100

    return round(growth, 2)

monthly_sales_joined_df["MoM_growth"] = monthly_sales_joined_df.apply(mom_growth, axis=1)
monthly_sales_joined_df.head()

In [ ]:
monthly_sales_joined_df["MoM_growth"].plot(figsize=(10, 3))

## transform()

In [ ]:
df_1.head()

In [ ]:
df_1.shape

In [ ]:
df_1.apply(sum)

In [ ]:
df_1["sales"].transform(lambda x: x * 100)

In [ ]:
df_1["sales"].apply(lambda x: x * 100)

In [ ]:
df_1.apply(lambda x: x * 100)

In [ ]:
df_1.transform(lambda x: x * 100)

In [ ]:
sales_df.groupby("order_year")["sales"].sum()

In [ ]:
sales_df["sales"].shape

In [ ]:
sales_df.groupby("order_year")["sales"].transform(sum)

### Question: Find the percentage total for each month of the year

In [ ]:
monthly_sales_df.head()

In [ ]:
total_sales_by_year = monthly_sales_df.groupby("order_year")["total_sales"].transform(sum)
total_sales_by_year

In [ ]:
sales_merged_df = pd.merge(monthly_sales_df, total_sales_by_year, left_index=True, right_index=True)
sales_merged_df.columns = ["total_sales", "total_yearly_sales"]
sales_merged_df.head()

In [ ]:
monthly_sales_df["total_yearly_sales_2"] = monthly_sales_df.groupby("order_year")["total_sales"].transform(sum)
monthly_sales_df.head()

In [ ]:
def pct_total(row):
    x, y = row["total_sales"], row["total_yearly_sales"]

    return round(x / y * 100, 2)

sales_merged_df["percentage_total"] = sales_merged_df.apply(pct_total, axis=1)
sales_merged_df

### Question: Find the months where the monthly sales were less then the average monthly sales for every year

Hint:

1. First find the monthly sales for each year
2. Use transform to find the average monthly sales for every year
3. Filter the data

In [ ]:
monthly_sales_df = sales_df.groupby(["order_year", "order_month"]).agg(
    total_sales= ("sales", "sum")
).reset_index()
 
monthly_sales_df["avg_monthly_sales"] = monthly_sales_df.groupby("order_year")["total_sales"].transform("mean")
 
result_df = monthly_sales_df[monthly_sales_df["total_sales"] < monthly_sales_df["avg_monthly_sales"]]
result_df

In [ ]:
#1. First find the monthly sales for each year
 
sales_df['order_month'] = sales_df['order_date'].dt.month
sales_df
 
sales_df['total_monthly_sales'] = sales_df.groupby(['order_year','order_month'])['sales'].transform('sum')
sales_df.head(1)
 
#2. Use transform to find the average monthly sales for every year
 
sales_df['avg_monthly_sales_per_year'] = sales_df.groupby('order_year')['total_monthly_sales'].transform('mean')
sales_df.head(1)
 
#3. Find the months
 
low_sales_df = sales_df[sales_df['total_monthly_sales'] < sales_df['avg_monthly_sales_per_year']]
low_sales_df.head(1)

final_months = low_sales_df[['order_year', 'order_month', 'total_monthly_sales', 'avg_monthly_sales_per_year']].drop_duplicates().sort_values(['order_year', 'order_month']).reset_index(drop=True)
final_months

### Question: Find the YoY Growth percentage for every product.

In [ ]:
monthly_sales_df = sales_df.groupby(["order_year", "product_name"]).agg(
    total_sales= ("sales", "sum")
).reset_index()
 
monthly_sales_df["yearly_sales"] = monthly_sales_df.groupby("product_name")["total_sales"].shift(1)
 
 
def pct_year(row):
    current, previous = row["total_sales"], row["yearly_sales"]
 
    return round(((current - previous)  / previous) * 100, 2)
 
monthly_sales_df["percentage"] = monthly_sales_df.apply(pct_year, axis = 1).fillna(0)
monthly_sales_df.sort_values(by=["product_name", "order_year"]) \
    .loc[:, ["product_name", "order_year", "total_sales", "yearly_sales", "percentage"]]

### Question: Out of total revenue generated, what percentage of revenue is from 'Wonka Bars'?

In [ ]:
# The following code adds a column called product category.
# The logic is if product_name contains "Wonka Bar" then the category is "Wonka Bar" else the category is "Other"
# We have to check the value of product_name in every row so to achieve this we are using apply
sales_df["product_category"] = sales_df.apply(lambda row: "Wonka Bar" if "Wonka Bar" in row["product_name"] else "Other", axis=1)

# The sign ~ means 'not'. i.e., True becomes False and vice-versa. So in this case, we are filtering product_name not containing "Wonka Bar"
sales_df[~sales_df["product_category"].str.contains("Wonka Bar")].head()

## Series.map()

In [ ]:
prodcuts_alternate_names_dict = {
    "Wonka Bar - Nutty Crunch Surprise" : "Wonka Bar",
    "Wonka Bar - Fudge Mallows" : "Wonka Bar",
    "Wonka Bar -Scrumdiddlyumptious" : "Wonka Bar",
    "Wonka Bar - Milk Chocolate" : "Wonka Bar",
    "Wonka Bar - Triple Dazzle Caramel" : "Wonka Bar",
    "Laffy Taffy" : "Taffy",
    "SweeTARTS" : "Tarts",
    "Nerds" : "Other",
    "Fun Dip" : "Other",
    "Fizzy Lifting Drinks" : "Other",
    "Everlasting Gobstopper" : "Other",
    "Hair Toffee" : "Toffee",
    "Lickable Wallpaper" : "Other",
    "Wonka Gum" : "Gum",
    "Kazookles" : "Other"
}

sales_df["product_alternate_names"] = sales_df["product_name"].map(prodcuts_alternate_names_dict)
sales_df[["product_name", "product_alternate_names"]].drop_duplicates()

In [ ]:
sales_df.head()

## Handling Duplicates

In [ ]:
sales_df["product_name"].drop_duplicates()

In [ ]:
sales_df["product_name"].drop_duplicates(keep="last")

In [ ]:
sales_df.columns

In [ ]:
sales_df[["division", "product_name", "ship_mode"]].drop_duplicates()

In [ ]:
sales_df.drop_duplicates(subset="order_date", keep="last").sort_values(by="order_date")

In [ ]:
sales_df.drop_duplicates(subset=["order_date", "ship_mode"]).sort_values(by=["order_date", "ship_mode"])

## Handling null values

In [ ]:
monthly_sales_joined_df.fillna(0)

In [ ]:
monthly_sales_joined_df["MoM_growth"] = monthly_sales_joined_df["MoM_growth"].fillna(0)
monthly_sales_joined_df.shape